In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.base import BaseEstimator, TransformerMixin
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from sklearn.svm import SVC

df = pd.read_csv("data/cleaned_data.csv")

# Split into features and target (X, y)
X = df.drop('Heart_Disease', axis=1)
df['Heart_Disease'] = df['Heart_Disease'].replace({'Yes': 1, 'No': 0})
y=df['Heart_Disease']

# Split into training and testing sets
# stratify=y ensures class balance in both train and test sets (very important for classification).
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42     
)

# Binary features still in object type
binary_cols = ['Exercise', 'Skin_Cancer', 'Other_Cancer',
               'Depression', 'Arthritis', 'Smoking_History']

# Categorical Ordinal features
ordinal_cols = ['General_Health', 'Checkup', 'Age_Category', 'Sex', 'Diabetes']

# Custom order for ordinal encoder
ordinal_mapping = [
    ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'],             # General_Health
    ['Never', '5 or more years ago', 'Within the past 5 years',
     'Within the past 2 years', 'Within the past year'],            # Checkup
    ['18-24', '25-29', '30-34', '35-39', '40-44',
     '45-49', '50-54', '55-59', '60-64', '65-69',
     '70-74', '75-79', '80+'],                                      # Age_Category
    ['Female', 'Male'],                                             # Sex
    ['No', 'No, pre-diabetes or borderline diabetes',
     'Yes, but female told only during pregnancy', 'Yes']           # Diabetes
]

# Numerical columns (already include log-transformed ones)
num_cols = [
    'Height_(cm)', 'Weight_(kg)', 'BMI', 'Alcohol_Consumption',
    'Fruit_Consumption', 'Green_Vegetables_Consumption', 'FriedPotato_Consumption',
    'Weight_(kg)_log', 'BMI_log', 'Alcohol_Consumption_log', 'Fruit_Consumption_log',
    'Green_Vegetables_Consumption_log', 'FriedPotato_Consumption_log'
]

class BinaryEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mapping = {'Yes': 1, 'No': 0}
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.replace(self.mapping).astype(int)


# Pipeline for numeric data
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Pipeline for binary categorical data
binary_pipeline = Pipeline([
    ('binary_encoder', BinaryEncoder())
])

# Pipeline for ordinal features
ordinal_pipeline = Pipeline([
    ('ordinal_encoder', OrdinalEncoder(categories=ordinal_mapping))
])

# Full preprocessor
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('bin', binary_pipeline, binary_cols),
    ('ord', ordinal_pipeline, ordinal_cols)
])

# n_splits = 5, smote = False, models included- LogisticRegression, RandomForest, XGBoost, CatBoost
# Step 1: Load your cleaned data (assumed done)
# Step 2: train/test split (assumed done: X_train, X_test, y_train, y_test)
# Step 3-6: preprocessing pipeline ready and defined as 'preprocessor'

# Optional: set smote usage
use_smote = False  # change to True to enable SMOTE


# Define your models
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, iterations=30, random_seed=42)
    # 'SVM': SVC(probability=True, class_weight='balanced', random_state=42)
}

# Function to build pipeline for a model
def build_pipeline(model, use_smote=False):
    steps = [('preprocessing', preprocessor)]
    if use_smote:
        steps.append(('smote', SMOTE(random_state=42)))
    steps.append(('classifier', model))
    return ImbPipeline(steps=steps)

# Store results
results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Loop over models
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    start_time = time.time()
    
    pipeline = build_pipeline(model, use_smote=use_smote)

    # Step 7: Cross-validation (on training data only)
    print(" Starting cross-validation...")
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
    print(f"Average ROC AUC (CV): {scores.mean():.4f}")
    
    # Step 8: Fit on train data
    print(" Fitting model on full training data...")
    pipeline.fit(X_train, y_train)
    print(" Model training complete.")
    
    # Step 9: Predict probabilities on test data
    print(" Predicting probabilities on test data...")
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # Step 10: Threshold tuning & evaluation
    print("Threshold tuning results:")
    thresholds = np.arange(0.1, 0.9, 0.05)
    best_f1 = 0
    best_thresh = 0.5
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        print(f"Threshold: {thresh:.2f} | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    end_time = time.time()
    elapsed = end_time - start_time
    print(f" Time taken for {model_name}: {elapsed:.2f} seconds")
    
    results[model_name] = {'cv_auc':scores.mean(), 'best_threshold': best_thresh, 'best_f1': best_f1}

# Summary of best thresholds and F1 scores
print("\nSummary of best thresholds and F1 scores:")
for model_name, res in results.items():
    print(f"{model_name}: CV AUC ={res['cv_auc']:.4f}, Best Threshold = {res['best_threshold']}, Best F1 Score = {res['best_f1']:.3f}")

# n_splits = 5, smote = True, models included- LogisticRegression, RandomForest, XGBoost, CatBoost
# Step 1: Load your cleaned data (assumed done)
# Step 2: train/test split (assumed done: X_train, X_test, y_train, y_test)
# Step 3-6: preprocessing pipeline ready and defined as 'preprocessor'

# Optional: set smote usage
use_smote =   # change to True to enable SMOTE


# Define your models
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'CatBoost': CatBoostClassifier(verbose=0, iterations=30, random_seed=42)
    # 'SVM': SVC(probability=True, class_weight='balanced', random_state=42)
}

# Function to build pipeline for a model
def build_pipeline(model, use_smote=False):
    steps = [('preprocessing', preprocessor)]
    if use_smote:
        steps.append(('smote', SMOTE(random_state=42)))
    steps.append(('classifier', model))
    return ImbPipeline(steps=steps)

# Store results
results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Loop over models
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    start_time = time.time()
    
    pipeline = build_pipeline(model, use_smote=use_smote)

    # Step 7: Cross-validation (on training data only)
    print(" Starting cross-validation...")
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
    print(f"Average ROC AUC (CV): {scores.mean():.4f}")
    
    # Step 8: Fit on train data
    print(" Fitting model on full training data...")
    pipeline.fit(X_train, y_train)
    print(" Model training complete.")
    
    # Step 9: Predict probabilities on test data
    print(" Predicting probabilities on test data...")
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # Step 10: Threshold tuning & evaluation
    print("Threshold tuning results:")
    thresholds = np.arange(0.1, 0.9, 0.05)
    best_f1 = 0
    best_thresh = 0.5
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        print(f"Threshold: {thresh:.2f} | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    end_time = time.time()
    elapsed = end_time - start_time
    print(f" Time taken for {model_name}: {elapsed:.2f} seconds")
    
    results[model_name] = {'cv_auc':scores.mean(), 'best_threshold': best_thresh, 'best_f1': best_f1}

# Summary of best thresholds and F1 scores
print("\nSummary of best thresholds and F1 scores:")
for model_name, res in results.items():
    print(f"{model_name}: CV AUC ={res['cv_auc']:.4f}, Best Threshold = {res['best_threshold']}, Best F1 Score = {res['best_f1']:.3f}")

from sklearn.metrics import confusion_matrix, roc_auc_score
best_threshold = 0.7

# Refit pipeline (if needed)
pipeline = build_pipeline(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
pipeline.fit(X_train, y_train)

# Predict probabilities
y_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred_final = (y_proba >= best_threshold).astype(int)

# Metrics
print("Classification Report:")
print(classification_report(y_test, y_pred_final))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

import joblib

def save_model(model, filename='best_model.pkl'):
    joblib.dump(model, filename)
save_model(pipeline, "logreg_best.pkl")
